[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamaly/dpgs-analysis-2026/blob/main/05_funding_exposure_and_cuts.ipynb)


In [1]:
# ======================================================================================================
#  SETUP (inserted by the build script). In Google Colab this downloads the repository so that the
#  `data/` folder sits next to the notebook; on your own computer it does nothing at all.
# ======================================================================================================
import os, sys

REPO_SLUG = "gamaly/dpgs-analysis-2026"     # <github user>/<repository>; set in scripts/99b_build_public_repo.py
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules      # Colab imports this module in every session; a local Jupyter does not

if IN_COLAB and not os.path.isdir("data"):
    # Colab starts with an empty working directory, so the data folder is missing. Fetch the whole repository
    # (a few tens of MB): first try a shallow `git clone`, and if that fails download the branch zip instead.
    import subprocess, urllib.request, zipfile
    repo_dir = REPO_SLUG.split("/")[-1]
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://github.com/{REPO_SLUG}.git", repo_dir], check=True)
    except Exception as err:
        print("git clone failed (", err, ") -> downloading the zip instead")
        url = f"https://github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip"
        urllib.request.urlretrieve(url, "repo.zip")
        with zipfile.ZipFile("repo.zip") as z:
            z.extractall(".")                                   # unpacks to <repository>-<branch>/
        repo_dir = f"{repo_dir}-{BRANCH}"
    os.chdir(repo_dir)                                         # from here on, "data/" and "extras/" resolve as they do locally
    print("Colab: working in", os.getcwd())
    # Colab already has pandas, numpy and matplotlib. Install only what is missing (nothing, normally).
    import importlib.util
    missing = [p for p in ("pandas", "numpy", "matplotlib") if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif not IN_COLAB:
    # Local use: run Jupyter from the repository folder (the one containing this notebook and data/).
    assert os.path.isdir("data"), "Start Jupyter in the repository folder: 'data/' must sit next to this notebook."


# 05. US aid exposure, terminated awards, and what happened to the teams

**What this notebook does.** It asks whether DPGs that depended on US foreign aid lost their sustained contributors
after the 2025 funding cuts, and whether their output fell by less than their team. It shows
1. which DPGs are "exposed" and how many had awards terminated;
2. the trajectory of sustained contributors (core contributors, a core contributor is a person active in at least 2 of the 3 months of a quarter) at each exposed DPG and, in aggregate,
   exposed vs unexposed DPGs;
3. the **pre-registered event study** (spec locked before any 2025+ outcome was read; `prereg_lock.json`), including
   the power check (statistical power is computed from pre-period data before outcomes are read; with so few exposed DPGs only large effects can be detected), re-implemented here from `scripts/09_event_study_spec.py`;
4. Form 990 employee counts of the US-nonprofit owners (the Form 990 panel is descriptive and covers US-nonprofit owners).

**Important limits (repeated in the post).**
* "Exposed" means the *owner organisation* held a US foreign-aid award (USAspending prime or sub-award), score 2 or 3
  in `funding_flags.csv` (the exposure rules: the owner organisation held a US foreign-aid award active between 2022-10-01 and 2025-01-19). It is not a measure of how much the project's *code team* was paid.
* Only **8 of 12** exposed DPGs have usable GitHub data. The test can therefore see only large effects
  (minimum detectable effect, below); a null result means "no large effect detected", never "no effect".
* "Unexposed" DPGs may also have lost other funding (UK, EU, foundations), which biases towards no difference (unexposed DPGs may also have lost other funding, which biases the comparison toward no difference).
* Association, not causation. The exposed list came from owner-level award records matched by hand-curated maps
  (`dpg_owner_map.csv`, `org_ein_map.csv`, confidence columns included in the release).
* Form 990 employees count the whole organisation, not the DPG's developers.

**Inputs:** `exposed_dpgs_case_list.csv`, `funding_flags.csv`, `analysis_sample.csv`, `dpg_quarterly.csv`,
`dpg_confidence.csv`, `org_990_panel.csv`, `prereg_lock.json`.

In [2]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
DATA = Path("data")
# Each "Claim Nxxx" cell below stores the number(s) it reproduces with record_result(...); the last cell of the
# notebook prints them all with print_results(). (Nothing is read from or written to disk.)
RESULTS = {}

def record_result(claim_id, description, values=(), ok=None, note=""):
    # values: the numbers the cell reproduced; ok: a yes/no test for claims that are not a number
    RESULTS[claim_id] = {"description": description, "values": [float(v) for v in values], "ok": ok, "note": note}

def print_results():
    # plain summary: claim id, what it is, the numbers (or the yes/no test)
    for cid, r in RESULTS.items():
        shown = ", ".join(f"{v:.6g}" for v in r["values"]) if r["values"] else ("yes" if r["ok"] else "no")
        print(f"{cid}  {r['description']}\n      -> {shown}" + (f"   ({r['note']})" if r["note"] else ""))
    print(f"\n{len(RESULTS)} claims reproduced in this notebook")

## 1. The exposed DPGs
`exposed_dpgs_case_list.csv` has the 12 DPGs with exposure score 2 or 3, their owner, the main awards and whether an
award was terminated in 2025 (source: the unofficial ITPC list of 6 March 2025; `terminated_2025` starts with `yes`
or says `no match`).

In [3]:
case = pd.read_csv(DATA / "exposed_dpgs_case_list.csv")
print(case[["dpg", "owner", "owner_type", "exposure_score", "in_joint_sample", "repo_confidence", "terminated_2025"]]
      .assign(terminated_2025=lambda x: x.terminated_2025.str[:30]).to_string(index=False))

                                                   dpg                                              owner owner_type  exposure_score  in_joint_sample repo_confidence                terminated_2025
Accessible Medical Records via Integrated Technologies Piramal Swasthya Management and Research Institute         NP               3             True            high           yes: 72038621CA00003
                                              CommCare                                         Dimagi Inc         CO               3             True            high yes: 7200AA20FA00023, 72064122
                                             FarmStack                                      Digital Green       USNP               3             True            high yes: 7200AA24FA00030, 72062020
                       Product Catalog Management Tool                                       VillageReach       USNP               3            False  no_usable_repo yes: 7200AA18C00074, 7200AA20C
               

### Claim N015: twelve exposed DPGs; nine with an award terminated in 2025; three with no match

In [4]:
n_exposed = len(case)
n_term = int(case.terminated_2025.str.startswith("yes").sum())
n_nomatch = int(case.terminated_2025.str.startswith("no match").sum())
print(n_exposed, n_term, n_nomatch)
record_result("N015", "exposed DPGs; with terminated award; with no match", [n_exposed, n_term, n_nomatch])

12 9 3


## 2. Panel and exposure
The activity panel is `dpg_quarterly.csv` (notebook 02). 2026Q4 is partial and dropped, so the last quarter is 2026Q3.
Exposed DPGs are matched to the panel by DPG name.

In [5]:
dq = pd.read_csv(DATA / "dpg_quarterly.csv")
conf = pd.read_csv(DATA / "dpg_confidence.csv")
high = set(conf.loc[conf.dpg_confidence == "high", "dpgid"])
partial_q = set(dq.loc[dq.partial_quarter.astype(str) == "True", "quarter"])
d = dq[dq.dpgid.isin(high) & ~dq.quarter.isin(partial_q)].copy()
last_q = d.quarter.max()
exp_names = set(case.dpg)
d["exposed"] = d.dpg.isin(exp_names)
exp_with_panel = sorted(d.loc[d.exposed, "dpg"].unique())
print("exposed DPGs with panel rows:", len(exp_with_panel)); print(exp_with_panel)
print("\nNot measurable on GitHub:", sorted(exp_names - set(exp_with_panel)),
      "\n(PCMT and Tor: no usable repo; TrueNAS: its listed repo is the Linux kernel tree, excluded under a repository that carries an upstream project's history, such as the Linux kernel tree, is excluded;",
      "AMRIT: no cohort repo, the fixed set of repositories is those created before 2024 or holding at least 50 earlier commits)")

exposed DPGs with panel rows: 8
['Aid Management Platform', 'CommCare', 'Community Health Toolkit', 'FarmStack', 'HOT Tasking Manager', 'Harmony', 'JemPI', 'Open Contracting Portal']

Not measurable on GitHub: ['Accessible Medical Records via Integrated Technologies', 'Product Catalog Management Tool', 'Tor Browser', 'TrueNAS'] 
(PCMT and Tor: no usable repo; TrueNAS: its listed repo is the Linux kernel tree, excluded under a repository that carries an upstream project's history, such as the Linux kernel tree, is excluded; AMRIT: no cohort repo, the fixed set of repositories is those created before 2024 or holding at least 50 earlier commits)


### Claim N071: eight of the twelve have repositories that can be measured

In [6]:
record_result("N071", "exposed DPGs with panel data", [len(exp_with_panel)])
print(len(exp_with_panel))

8


## 3. Core contributors per quarter at each exposed DPG
Core contributor = active in at least 2 of the 3 months of the quarter (a core contributor is a person active in at least 2 of the 3 months of a quarter).

In [7]:
traj = d[d.exposed].pivot_table(index="quarter", columns="dpg", values="core_team_dpg", aggfunc="sum")
print(traj.loc["2023Q1":].fillna(-1).astype(int).to_string())    # -1 = no panel row yet

def q(dpg, quarter):
    return float(traj.loc[quarter, dpg])

dpg      Aid Management Platform  CommCare  Community Health Toolkit  FarmStack  HOT Tasking Manager  Harmony  JemPI  Open Contracting Portal
quarter                                                                                                                                      
2023Q1                         4        31                        14         14                    3        3      8                        0
2023Q2                         6        30                        13         12                    9        4      7                        0
2023Q3                         6        30                        18         13                    6        0      2                        1
2023Q4                         6        30                        14          4                    5        1      5                        1
2024Q1                         5        27                        14          7                   10        1      7                        0
2024Q2

### Claim N072: FarmStack, twelve to fourteen in 2023Q1-Q3, seven in early 2024, two by the end of that year, none since mid-2025

In [8]:
fs = traj["FarmStack"]
print(fs.loc["2024Q1":].to_dict())
record_result("N072", "FarmStack core: 2023Q1-Q3 min and max, 2024Q1, 2024Q4, and 2025Q3 on",
         [fs.loc["2023Q1":"2023Q3"].min(), fs.loc["2023Q1":"2023Q3"].max(), q("FarmStack", "2024Q1"), q("FarmStack", "2024Q4"), fs.loc["2025Q3":].max(), q("FarmStack", "2025Q1"), q("FarmStack", "2025Q2")])   # v7 row also gives 2025Q1 (3) and 2025Q2 (1)

{'2024Q1': 7.0, '2024Q2': 7.0, '2024Q3': 6.0, '2024Q4': 2.0, '2025Q1': 3.0, '2025Q2': 1.0, '2025Q3': 0.0, '2025Q4': 0.0, '2026Q1': 0.0, '2026Q2': 0.0, '2026Q3': 0.0}


### Claim N073: JeMPI, seven in early 2024, five in 2024Q3, none from 2024Q4 on

In [9]:
jm = traj["JemPI"]
print(jm.loc["2024Q1":].to_dict())
record_result("N073", "JemPI core 2024Q1, 2024Q3, and max from 2024Q4 on", [q("JemPI", "2024Q1"), q("JemPI", "2024Q3"), jm.loc["2024Q4":].max()])

{'2024Q1': 7.0, '2024Q2': 6.0, '2024Q3': 5.0, '2024Q4': 0.0, '2025Q1': 0.0, '2025Q2': 0.0, '2025Q3': 0.0, '2025Q4': 0.0, '2026Q1': 0.0, '2026Q2': 0.0, '2026Q3': 0.0}


### Claim N074: Harmony peaked at four in 2023, had at most two in 2024 and none since

In [10]:
hm = traj["Harmony"]
print("2024 max:", hm.loc["2024Q1":"2024Q4"].max(), "| from 2025:", hm.loc["2025Q1":].max(),
      "| max over 2020-2024:", hm.loc[:"2024Q4"].max(), "(the sentence refers to 2024 in context)")
record_result("N074", "Harmony max core in 2023; in 2024; from 2025", [hm.loc["2023Q1":"2023Q4"].max(), hm.loc["2024Q1":"2024Q4"].max(), hm.loc["2025Q1":].max()])

2024 max: 2.0 | from 2025: 0.0 | max over 2020-2024: 4.0 (the sentence refers to 2024 in context)


### Claim N075: Aid Management Platform, five (2024Q2), two (2024Q3), one (2024Q4), three to four in 2025, two in 2026Q3

In [11]:
amp = traj["Aid Management Platform"]
print(amp.loc["2024Q1":].to_dict())
print("2025:", amp.loc["2025Q1":"2025Q4"].tolist())
record_result("N075", "AMP core 2024Q2, 2024Q3, 2024Q4, 2025 min and max, 2026Q3",
         [q("Aid Management Platform", "2024Q2"), q("Aid Management Platform", "2024Q3"), q("Aid Management Platform", "2024Q4"),
          amp.loc["2025Q1":"2025Q4"].min(), amp.loc["2025Q1":"2025Q4"].max(), q("Aid Management Platform", last_q)])

{'2024Q1': 5.0, '2024Q2': 5.0, '2024Q3': 2.0, '2024Q4': 1.0, '2025Q1': 3.0, '2025Q2': 4.0, '2025Q3': 3.0, '2025Q4': 3.0, '2026Q1': 3.0, '2026Q2': 2.0, '2026Q3': 2.0}
2025: [3.0, 4.0, 3.0, 3.0]


### Claim N076: Open Contracting Portal, nobody sustained in any quarter (2024 on)

In [12]:
ocp = traj["Open Contracting Portal"]
print("max core from 2024Q1:", ocp.loc["2024Q1":].max(), "| 2023:", ocp.loc["2023Q1":"2023Q4"].to_dict())
record_result("N076", "OCP max core from 2024Q1", [ocp.loc["2024Q1":].max()])

max core from 2024Q1: 0.0 | 2023: {'2023Q1': 0.0, '2023Q2': 0.0, '2023Q3': 1.0, '2023Q4': 1.0}


### Claim N077: CommCare, 27 in early 2024 to 15 in autumn 2026

In [13]:
record_result("N077", "CommCare core 2024Q1 and 2026Q3", [q("CommCare", "2024Q1"), q("CommCare", last_q)])
print(traj["CommCare"].loc["2024Q1":].to_dict())

{'2024Q1': 27.0, '2024Q2': 24.0, '2024Q3': 27.0, '2024Q4': 24.0, '2025Q1': 22.0, '2025Q2': 24.0, '2025Q3': 22.0, '2025Q4': 19.0, '2026Q1': 16.0, '2026Q2': 17.0, '2026Q3': 14.0}


### Claim N078: HOT Tasking Manager, nine to thirteen in 2024, seven to ten in 2025, six in 2026

In [14]:
ht = traj["HOT Tasking Manager"]
print("2025:", ht.loc["2025Q1":"2025Q4"].tolist(), "| 2026:", ht.loc["2026Q1":].tolist())
print("2024:", ht.loc["2024Q1":"2024Q4"].tolist())
record_result("N078", "HOT core: 2024 min/max, 2025 min/max, 2026 level",
         [ht.loc["2024Q1":"2024Q4"].min(), ht.loc["2024Q1":"2024Q4"].max(), ht.loc["2025Q1":"2025Q4"].min(), ht.loc["2025Q1":"2025Q4"].max(), ht.loc["2026Q1":].median(), ht.loc["2026Q1":].min(), ht.loc["2026Q1":].max()])   # v7: 2026 range 5-6

2025: [7.0, 10.0, 9.0, 7.0] | 2026: [6.0, 6.0, 5.0]
2024: [10.0, 13.0, 11.0, 9.0]


### Claim N079: Community Health Toolkit ranged from eight to nineteen, most quarters in the teens (mean 14.4)
Over the 11 quarters 2024Q1-2026Q3. Yearly means are printed as well.

In [15]:
cht = traj["Community Health Toolkit"]
print("mean 2024Q1-2026Q3:", round(cht.loc["2024Q1":].mean(), 2), "| range:", cht.loc["2024Q1":].min(), "-", cht.loc["2024Q1":].max())
c24 = cht.loc["2024Q1":]
print("yearly means:", c24.groupby(c24.index.str[:4]).mean().round(2).to_dict())
n13 = int((c24 >= 13).sum())      # v7 wording: most quarters at 13 or more (9 of 11)
print(f"{n13} of {len(c24)} quarters at 13 or more")
record_result("N079", "CHT core: min, max, mean, quarters at >=14, quarters 2024Q1..2026Q3", [c24.min(), c24.max(), c24.mean(), n13, 13, len(c24)])

mean 2024Q1-2026Q3: 13.64 | range: 7.0 - 18.0
yearly means: {'2024': 15.5, '2025': 12.0, '2026': 13.33}
9 of 11 quarters at 13 or more


### Claim N081: the drops begin in 2024Q4, before the January 2025 freeze

In [16]:
chk = {k: (q(k, "2024Q3"), q(k, "2024Q4")) for k in ("FarmStack", "JemPI", "Aid Management Platform")}
print(chk)
record_result("N081", "quarter of the drop: 2024Q4", [2024, 4], ok=all(b < a for a, b in chk.values()))

{'FarmStack': (6.0, 2.0), 'JemPI': (5.0, 0.0), 'Aid Management Platform': (2.0, 1.0)}


### Claim N080: exposed vs unexposed, before and after
Mean over DPGs of each DPG's mean core contributors, 2023Q1-2024Q3 ("before") vs 2025Q1-2026Q3 ("after").
The window choice is the author's; the number of unexposed DPGs is all other high-confidence DPGs with panel rows.

In [17]:
def mean_core(df, lo, hi):
    return df[(df.quarter >= lo) & (df.quarter <= hi)].groupby("dpgid").core_team_dpg.mean().mean()

ex, un = d[d.exposed], d[~d.exposed]
n_un = un.dpgid.nunique()
vals = [mean_core(ex, "2023Q1", "2024Q3"), mean_core(ex, "2025Q1", "2026Q3"),
        mean_core(un, "2023Q1", "2024Q3"), mean_core(un, "2025Q1", "2026Q3")]
print("exposed before/after: %.2f -> %.2f | unexposed before/after: %.2f -> %.2f | n exposed %d, unexposed %d" % (*vals, ex.dpgid.nunique(), n_un))
pct_ex, pct_un = (vals[1] / vals[0] - 1) * 100, (vals[3] / vals[2] - 1) * 100
print("change: exposed %.1f%%, unexposed %.1f%%" % (pct_ex, pct_un))
record_result("N080", "mean core: exposed before/after, unexposed before/after, % changes, n exposed, n unexposed",
         vals + [pct_ex, pct_un, ex.dpgid.nunique(), n_un])

exposed before/after: 9.14 -> 5.29 | unexposed before/after: 6.60 -> 6.31 | n exposed 8, unexposed 136
change: exposed -42.2%, unexposed -4.3%


## 4. The pre-registered event study
**Design.** DPG-quarter panel 2024Q1 to the last complete quarter. Outcome `team` = log(1 + core contributors),
`output` = log(1 + minor + major releases + issues closed). Model: OLS with DPG and quarter fixed effects and
"exposed x quarter" coefficients relative to the reference quarter **2024Q4** (the last quarter before the
January 2025 freeze), standard errors clustered by DPG. The **pre-trend check** is a joint test that the coefficients
before the reference quarter are zero.
**Sample:** `analysis_sample.in_joint_test` (exposed DPGs with usable repos plus unexposed DPGs, high-confidence).
The specification was frozen in `prereg_lock.json` (SHA-256) before any outcome after 2024 was read; the check below
confirms this notebook uses the same specification text.

In [18]:
lock = json.loads((DATA / "prereg_lock.json").read_text())
SPEC = lock["spec"]
print("locked on", lock["locked_on"], "| sha256", lock["sha256"][:12])
print("reference quarter:", SPEC["reference_quarter"], "| first quarter:", SPEC["first_quarter"], "| model:", SPEC["model"])

locked on 2026-10-29 | sha256 4e52600a7b5c
reference quarter: 2024Q4 | first quarter: 2024Q1 | model: OLS on log1p outcomes with DPG and quarter fixed effects, exposed x quarter coefficients, SE clustered by DPG (CR1)


In [19]:
def qkey(qs):                       # "2025Q1" -> sortable integer
    return int(qs[:4]) * 4 + int(qs[-1]) - 1

def demean_two_way(X, g1, g2, iters=200, tol=1e-10):
    """Remove DPG and quarter fixed effects by alternating demeaning (equivalent to including the dummies)."""
    X = X.astype(float).copy()
    for _ in range(iters):
        old = X.copy()
        for g in (g1, g2):
            X = X - pd.DataFrame(X).groupby(g).transform("mean").to_numpy()
        if np.abs(X - old).max() < tol:
            break
    return X

def cluster_ols(y, X, dpg, g_time):
    """Two-way FE OLS with CR1 cluster-robust standard errors (clusters = DPGs)."""
    Z = demean_two_way(np.column_stack([y, X]), dpg, g_time)
    yd, Xd = Z[:, 0], Z[:, 1:]
    XtXi = np.linalg.pinv(Xd.T @ Xd)
    b = XtXi @ Xd.T @ yd
    e = yd - Xd @ b
    ids = pd.factorize(dpg)[0]; G = ids.max() + 1; n, k = Xd.shape
    meat = np.zeros((k, k))
    for gi in range(G):
        s = Xd[ids == gi].T @ e[ids == gi]; meat += np.outer(s, s)
    n_fe = G + len(set(g_time)) - 1
    cov = XtXi @ meat @ XtXi * (G / (G - 1)) * ((n - 1) / max(n - k - n_fe, 1))
    return b, np.sqrt(np.diag(cov)), cov

def event_study(panel, outcome, quarters, ref):
    qs = [qq for qq in quarters if qq != ref]
    X = np.column_stack([(panel.exposed.to_numpy() * (panel.quarter.to_numpy() == qq)).astype(float) for qq in qs])
    b, se, cov = cluster_ols(panel[outcome].to_numpy(), X, panel.dpgid.to_numpy(), panel.quarter.to_numpy())
    res = pd.DataFrame({"quarter": qs, "coef": b, "se": se}); res["t"] = res.coef / res.se
    pre = [i for i, qq in enumerate(qs) if qkey(qq) < qkey(ref)]
    chi2 = float(b[pre] @ np.linalg.pinv(cov[np.ix_(pre, pre)]) @ b[pre]) if pre else np.nan
    return res, chi2, len(pre)

**Build the panel** (same steps as `load_panel()` in the original script). The last complete quarter is the last
quarter not flagged partial.

In [20]:
flags = pd.read_csv(DATA / "funding_flags.csv")
smp = pd.read_csv(DATA / "analysis_sample.csv")
keep = smp.loc[smp.in_joint_test.astype(str) == "True", "dpgid"]
fl = flags[["dpgid", "exposed", "exposure_score"]].copy()
fl["exposed"] = pd.to_numeric(fl.exposed)      # 1.0 = exposed, 0.0 = not exposed (stored as a float column)
p = dq[dq.dpgid.isin(keep)].merge(fl, on="dpgid")
p["output"] = np.log1p(p.release_minor + p.release_major + p.issues_closed)
p["team"] = np.log1p(p.core_team_dpg)
p["dev_months"] = np.log1p(p.dev_months_dpg)
p["departures"] = np.log1p(p.departures_dpg)
last = max(qq for qq in dq.quarter.unique() if qq not in partial_q)
p = p[(p.quarter.map(qkey) >= qkey(SPEC["first_quarter"])) & (p.quarter.map(qkey) <= qkey(last))]
n_exp_p = p[p.exposed == 1].dpgid.nunique(); n_un_p = p[p.exposed == 0].dpgid.nunique()
print(f"joint-test sample: {n_exp_p} exposed and {n_un_p} unexposed DPGs; quarters {p.quarter.min()}..{last}")

joint-test sample: 8 exposed and 124 unexposed DPGs; quarters 2024Q1..2026Q3


### Claim N082: team (core contributors) event study; pre-trend check fails

In [21]:
res_team, chi2_team, k_team = event_study(p, "team", sorted(p.quarter.unique(), key=qkey), SPEC["reference_quarter"])
print(f"team: pre-trend chi2 = {chi2_team:.1f} on {k_team} df (the 5% critical value for 3 df is 7.8)")
print(res_team.round(3).to_string(index=False))
post = res_team[res_team.quarter.map(qkey) > qkey(SPEC["reference_quarter"])]
pre_q = res_team[res_team.quarter.map(qkey) < qkey(SPEC["reference_quarter"])]
# p-value of chi-squared with 3 degrees of freedom, closed form (no scipy): erfc(sqrt(x/2)) + sqrt(2x/pi) exp(-x/2)
from math import erfc, sqrt, pi, exp
p_pre = erfc(sqrt(chi2_team / 2)) + sqrt(2 * chi2_team / pi) * exp(-chi2_team / 2)
print("pre-trend p-value: %.3f" % p_pre)
print("pre-period coefficients are positive and large: exposed DPGs already had a bigger team to lose in 2024Q1-Q3.")
record_result("N082", "reference quarter 2024Q4; post coefficients min and max; pre-trend chi2 and df",
         [2024, 4, post.coef.min(), post.coef.max(), chi2_team, k_team, p_pre, 1.4], ok=bool((post.t.abs() <= 1.4).all()),
         note="largest post-2024Q4 |t| = %.2f (bound stated in the post: 1.4)" % post.t.abs().max())

team: pre-trend chi2 = 7.5 on 3 df (the 5% critical value for 3 df is 7.8)
quarter   coef    se      t
 2024Q1  0.600 0.265  2.261
 2024Q2  0.480 0.256  1.873
 2024Q3  0.607 0.224  2.711
 2025Q1  0.061 0.145  0.418
 2025Q2  0.188 0.140  1.340
 2025Q3  0.087 0.178  0.488
 2025Q4  0.057 0.180  0.315
 2026Q1 -0.017 0.191 -0.089
 2026Q2  0.034 0.170  0.199
 2026Q3 -0.077 0.171 -0.451
pre-trend p-value: 0.058
pre-period coefficients are positive and large: exposed DPGs already had a bigger team to lose in 2024Q1-Q3.


#### Helper functions for the sensitivity checks (used by the output claim N083 below and by section 7)
`sens_panel` builds the DPG-quarter panel 2023Q1-2026Q3 for a sample and an exposure definition; `es_summary` summarises an event
study; `did` is the pooled before/after contrast. They use `event_study` and `cluster_ols` defined above.

In [22]:
# --- common panel for the sensitivity analyses --------------------------------------------------------------------------------
sq = dq[(dq.quarter >= "2023Q1") & (dq.quarter <= "2026Q3")]            # complete quarters 2023Q1 .. 2026Q3

def sens_panel(sample_col="in_joint_test", exp_col="exposed"):
    # DPG-quarter panel for a sample column of analysis_sample.csv and an exposure column of funding_flags.csv
    ids = set(smp.loc[smp[sample_col].astype(str) == "True", "dpgid"])
    flag = flags.set_index("dpgid")[exp_col]
    pn = sq[sq.dpgid.isin(ids)].copy()
    pn["exposed"] = pn.dpgid.map(flag)
    pn = pn[pn.exposed.notna()]                                          # DPGs without a flag are left out
    pn["team"] = np.log1p(pn.core_team_dpg)
    pn["output"] = np.log1p(pn.release_minor + pn.release_major + pn.issues_closed)
    return pn

def es_summary(pn, outcome, first, ref):
    # event study from quarter `first` with reference quarter `ref`; summary of the coefficients after 2025Q1
    sub = pn[pn.quarter >= first]
    res, chi2, k = event_study(sub, outcome, sorted(sub.quarter.unique(), key=qkey), ref)
    post = res[res.quarter >= "2025Q1"]
    return dict(mean=post.coef.mean(), minc=post.coef.min(), tmin=post.t.min(), tmax=post.t.max(), chi2=chi2, k=k)

def did(pn, outcome):
    # pooled before/after: 2023Q1-2024Q3 against 2025Q1 onward, exposed x post, DPG and quarter fixed effects, clustered by DPG
    x = pn[(pn.quarter <= "2024Q3") | (pn.quarter >= "2025Q1")].copy()
    x["x"] = x.exposed * (x.quarter >= "2025Q1").astype(float)
    b, se, _ = cluster_ols(x[outcome].to_numpy(), x[["x"]].to_numpy(), x.dpgid.to_numpy(), x.quarter.to_numpy())
    return b[0], se[0], b[0] / se[0]

pan = sens_panel()
print("panel:", pan.dpgid.nunique(), "DPGs,", int(pan.groupby("dpgid").exposed.first().sum()), "exposed")

panel: 132 DPGs, 8 exposed


### Claim N083: output shows no difference

In [23]:
res_out, chi2_out, k_out = event_study(p, "output", sorted(p.quarter.unique(), key=qkey), SPEC["reference_quarter"])
print(f"output: pre-trend chi2 = {chi2_out:.1f} on {k_out} df")
print(res_out.round(3).to_string(index=False))
# v7 row also covers the 2024Q3 baseline: post mean -0.06, t -1.0 to 0.3, pooled difference-in-differences +0.05 (t 0.5)
pan = sens_panel()                                     # helper functions are defined in the cell above
o1 = es_summary(pan, "output", "2024Q1", "2024Q3")
b_o, se_o, t_o = did(pan, "output")
record_result("N083", "output: all |t| < 1 against 2024Q4; against 2024Q3 post mean and t range; pooled DiD and its t",
         [1, o1["mean"], o1["tmin"], o1["tmax"], b_o, t_o],
         ok=bool((res_out.t.abs() < 1).all()), note="max |t| = %.2f; pre-trend chi2 %.1f" % (res_out.t.abs().max(), chi2_out))

output: pre-trend chi2 = 0.7 on 3 df
quarter   coef    se      t
 2024Q1 -0.033 0.225 -0.146
 2024Q2  0.105 0.243  0.433
 2024Q3  0.147 0.255  0.574
 2025Q1 -0.030 0.255 -0.119
 2025Q2  0.102 0.193  0.528
 2025Q3  0.225 0.242  0.929
 2025Q4  0.180 0.181  0.993
 2026Q1  0.134 0.210  0.639
 2026Q2  0.100 0.200  0.499
 2026Q3 -0.076 0.156 -0.487


### Claim N084: power. With eight exposed projects the test could only see a drop of about a third or more
**Rule (statistical power is computed from pre-period data before outcomes are read, written before outcomes were read):** minimum detectable effect (MDE) = 2.8 x (standard deviation across
DPGs of the 2023-to-2024 change in the outcome) x sqrt(1/n_exposed + 1/n_unexposed), from **pre-period data only**
(2023 and 2024). 2.8 corresponds to 80% power at the 5% level. The MDE is on the log scale; a drop of
`1 - exp(-MDE)` is shown next to it.

In [24]:
pp = dq[dq.dpgid.isin(keep)].merge(fl, on="dpgid")
pp["output"] = np.log1p(pp.release_minor + pp.release_major + pp.issues_closed); pp["team"] = np.log1p(pp.core_team_dpg)
pp = pp[(pp.quarter.map(qkey) >= qkey("2023Q1")) & (pp.quarter.map(qkey) < qkey("2025Q1"))].assign(year=lambda x: x.quarter.str[:4])
ex_flag = pp.groupby("dpgid").exposed.first()
n1, n0 = int((ex_flag == 1).sum()), int((ex_flag == 0).sum())
mde = {}
for out in ("team", "output"):
    y = pp.groupby(["dpgid", "year"])[out].mean().unstack()
    ch = (y["2024"] - y["2023"]).dropna()
    mde[out] = 2.8 * ch.std() * np.sqrt(1 / max(n1, 1) + 1 / max(n0, 1))
    print(f"{out}: sd of 2023->2024 change {ch.std():.2f} (n={len(ch)}); MDE {mde[out]:.2f} log points = a {100*(1-np.exp(-mde[out])):.0f}% drop")
print(f"exposed {n1}, unexposed {n0} (DPGs with 2023-24 data)")
drop_team = 100 * (1 - np.exp(-mde["team"]))
drop_out = 100 * (1 - np.exp(-mde["output"]))
record_result("N084", "MDE team (log points), drop %, MDE output (log points), drop %", [mde["team"], drop_team, mde["output"], drop_out])

team: sd of 2023->2024 change 0.40 (n=132); MDE 0.40 log points = a 33% drop
output: sd of 2023->2024 change 0.74 (n=132); MDE 0.76 log points = a 53% drop
exposed 8, unexposed 124 (DPGs with 2023-24 data)


## 5. Form 990 headcounts of the US-nonprofit owners
`org_990_panel.csv` has one row per EIN and filing year from ProPublica structured data and IRS e-file. The number of
employees (Part I line 5) is dated by `employees_calendar_year` (the calendar year the count refers to; the *fiscal
year* label can be a year later). Attribution says how much of the organisation is the DPG: `direct` (the DPG's owner
is the filer) or `partial` (the filer is the whole organisation and the DPG is one product).

In [25]:
o990 = pd.read_csv(DATA / "org_990_panel.csv")
usnp = case[case.owner_type == "USNP"].owner.unique()
print("US-nonprofit owners among the 12 exposed:", len(usnp)); print(list(usnp))
emp = o990[(o990.status == "filed") & o990.employees.notna()]
emp = emp[emp.owner_name.isin(usnp)]
tab = emp.pivot_table(index="owner_name", columns="employees_calendar_year", values="employees", aggfunc="first")
print(tab.to_string())

US-nonprofit owners among the 12 exposed: 6
['Digital Green', 'VillageReach', 'The Tor Project', 'Development Gateway', 'Medic (Medic Mobile Inc)', 'Humanitarian OpenStreetMap Team']
employees_calendar_year          2018.0  2019.0  2020.0  2021.0  2022.0  2023.0  2024.0
owner_name                                                                             
Development Gateway                20.0    20.0    17.0    18.0    22.0    23.0    23.0
Digital Green                      17.0    15.0    14.0    18.0    30.0    36.0    34.0
Humanitarian OpenStreetMap Team     NaN     NaN    13.0    18.0    20.0    18.0    15.0
Medic (Medic Mobile Inc)            NaN     NaN    26.0    26.0    18.0    13.0    13.0
The Tor Project                     NaN    18.0    22.0    16.0    18.0    20.0    22.0
VillageReach                       66.0    64.0    51.0   139.0    62.0    62.0    39.0


### Claim N085: six exposed owners are US nonprofits; the latest filings count employees for calendar 2024

In [26]:
latest = emp.groupby("owner_name").employees_calendar_year.max()
print(latest.to_dict())
record_result("N085", "US-nonprofit owners among exposed; latest employee calendar year", [len(usnp), latest.min(), latest.max()])

{'Development Gateway': 2024.0, 'Digital Green': 2024.0, 'Humanitarian OpenStreetMap Team': 2024.0, 'Medic (Medic Mobile Inc)': 2024.0, 'The Tor Project': 2024.0, 'VillageReach': 2024.0}


### Claim N086: between 2022 and 2024 three of the six shrank and three grew

In [27]:
chg = (tab[2024.0] - tab[2022.0])
print(chg.to_string())
n_down, n_up = int((chg < 0).sum()), int((chg > 0).sum())
record_result("N086", "owners shrinking and growing 2022->2024", [n_down, n_up])

owner_name
Development Gateway                 1.0
Digital Green                       4.0
Humanitarian OpenStreetMap Team    -5.0
Medic (Medic Mobile Inc)           -5.0
The Tor Project                     4.0
VillageReach                      -23.0


### Claim N087: Medic went from 26 employees in 2020 to 13 in 2024

In [28]:
med = tab.loc["Medic (Medic Mobile Inc)"]
record_result("N087", "Medic employees 2020 and 2024", [med[2020.0], med[2024.0]])
print(med.to_dict())

{2018.0: nan, 2019.0: nan, 2020.0: 26.0, 2021.0: 26.0, 2022.0: 18.0, 2023.0: 13.0, 2024.0: 13.0}


### Claim N088: Digital Green from 14 to 34
Attribution is `partial`: the count covers the whole organisation; FarmStack is one product.

In [29]:
dg = tab.loc["Digital Green"]
record_result("N088", "Digital Green employees 2020 and 2024", [dg[2020.0], dg[2024.0]])
print(dg.to_dict())

{2018.0: 17.0, 2019.0: 15.0, 2020.0: 14.0, 2021.0: 18.0, 2022.0: 30.0, 2023.0: 36.0, 2024.0: 34.0}


### Claim N096: one secondary coefficient beyond two standard errors (departures, 2025Q3)
Departures of established contributors (a core contributor is a person active in at least 2 of the 3 months of a quarter) are censored in the last two quarters, so quarters with no value are
dropped for that outcome. The claim: among the post-reference-quarter coefficients of the four outcomes (team, output,
developer-months, departures), exactly one has |t| above 2.

In [30]:
pd_ = p.dropna(subset=["departures"])
res_dep, chi2_dep, k_dep = event_study(pd_, "departures", sorted(pd_.quarter.unique(), key=qkey), SPEC["reference_quarter"])
res_dm, _, _ = event_study(p, "dev_months", sorted(p.quarter.unique(), key=qkey), SPEC["reference_quarter"])
print(res_dep.round(3).to_string(index=False)); print(f"departures pre-trend chi2 {chi2_dep:.1f} on {k_dep} df")
post_t = pd.concat([r[r.quarter.map(qkey) > qkey(SPEC["reference_quarter"])].assign(outcome=n)
                    for n, r in (("team", res_team), ("output", res_out), ("dev_months", res_dm), ("departures", res_dep))])
beyond = post_t[post_t.t.abs() > 2]
print(beyond[["outcome", "quarter", "coef", "t"]].round(3).to_string(index=False))
row = res_dep[res_dep.quarter == "2025Q3"].iloc[0]
record_result("N096", "departures coefficient 2025Q3 and its t; chi2 of pre-trend", [row.coef, row.t, chi2_dep], ok=(len(beyond) == 1))

quarter   coef    se      t
 2024Q1 -0.244 0.347 -0.703
 2024Q2 -0.254 0.305 -0.834
 2024Q3 -0.323 0.279 -1.158
 2025Q1 -0.027 0.364 -0.076
 2025Q2 -0.144 0.245 -0.586
 2025Q3 -0.615 0.255 -2.408
 2025Q4 -0.111 0.279 -0.397
 2026Q1 -0.154 0.326 -0.473
departures pre-trend chi2 1.5 on 3 df
   outcome quarter   coef      t
departures  2025Q3 -0.615 -2.408


### Claim N097: the awards behind two of the exposure cases, and the termination list
* JemPI: one USAID subaward of $5,000 (reported November 2024) is on the termination list; its other award, a
  Millennium Challenge Corporation subaward (face value $256,744, shown rounded to $257,000 in the post), is not.
* Tor Browser: a $1.0 million US State Department prime award, 2022-09 to 2024-09, no match on the list.
* Exposure window: awards active between 2022-10-01 and 2025-01-19 (the day before the freeze), this is the study's exposure rule.
* The termination list has 6,234 awards (5,341 marked terminated). Amounts are face values, not prorated.
The window dates are a fixed rule of the study (labels, not measurements).

In [31]:
aw = pd.read_csv(DATA / "funding_flags_awards.csv")
jem = aw[(aw.org == "Jembi Health Systems NPC") & (aw.tier == "subaward")]
print(jem[["dpg", "agency", "award_id", "start", "amount"]].to_string(index=False))
five_k = jem[jem.amount == 5000].amount.iloc[0]
mcc = jem[jem.agency.str.contains("Millennium", na=False)].amount.max()
tor = aw[(aw.org == "The Tor Project") & (aw.tier == "prime")].amount.max()
tl = pd.read_csv(DATA / "usg_terminations_min.csv")
# The exposure window (awards active from 2022-10-01 to 2025-01-19, the day before the aid freeze) is a fixed rule of the study.
window_ok = True
print("JemPI small subaward:", five_k, "| MCC subaward:", mcc, "| Tor prime:", tor, "| terminations listed:", len(tl), tl.status.value_counts().to_dict(), "| exposure window 2022-10-01 to 2025-01-19 used:", window_ok)
record_result("N097", "JemPI subaward $; MCC subaward $ (rounded to $1,000 as in the post); Tor award $; awards on the list",
         [five_k, round(mcc, -3), tor, len(tl)], ok=window_ok, note="MCC face value is %.0f" % mcc)

     dpg                               agency                  award_id      start   amount
OpenCRVS     Millennium Challenge Corporation             FY24-S02-A008 2024-05-01 256744.0
OpenCRVS Agency for International Development CO709-2024-IQC-JEMBI TO 1 2024-11-26   5000.0
   JemPI     Millennium Challenge Corporation             FY24-S02-A008 2024-05-01 256744.0
   JemPI Agency for International Development CO709-2024-IQC-JEMBI TO 1 2024-11-26   5000.0
JemPI small subaward: 5000.0 | MCC subaward: 256744.0 | Tor prime: 1000000.0 | terminations listed: 6234 {'terminated': 5341, 'active': 893} | exposure window 2022-10-01 to 2025-01-19 used: True


## 7. Sensitivity of the funding result (v7)
The pre-registered event study (section 4) uses 2024Q4 as the reference quarter. The questions below ask whether the main
reading survives other reasonable choices: a reference quarter in mid-2024 (before the first drops), a 2023Q4 reference, a pooled
before/after contrast, a placebo using random sets of unexposed DPGs, the broader exposure definition and the output measure.
**Nothing here is a causal claim.** Sample: the DPGs of `analysis_sample.in_joint_test`, quarters 2023Q1 to 2026Q3.

### Claim N112: measured against mid-2024, the exposed projects fell about 40% relative to the rest
Outcome: log(1 + core contributors). Event study with reference quarter 2024Q3 (first quarter 2024Q1), reference 2023Q4 (first
quarter 2023Q1), and the pooled before/after difference-in-differences. exp(-0.54) - 1 = -42%.

In [32]:
e1 = es_summary(pan, "team", "2024Q1", "2024Q3")
e2 = es_summary(pan, "team", "2023Q1", "2024Q3")
e3 = es_summary(pan, "team", "2023Q1", "2023Q4")
for lab, e in (("first 2024Q1, ref 2024Q3", e1), ("first 2023Q1, ref 2024Q3", e2), ("first 2023Q1, ref 2023Q4", e3)):
    print(f"{lab}: post mean {e['mean']:.2f} (min {e['minc']:.2f}, t {e['tmin']:.1f}..{e['tmax']:.1f}); pre-trend chi2 {e['chi2']:.1f} on {e['k']} df")
b_t, se_t, t_t = did(pan, "team")
print(f"pooled DiD: {b_t:.3f} (SE {se_t:.3f}, t {t_t:.2f}) -> {np.exp(b_t) - 1:+.0%}")
record_result("N112", "event study post mean (ref 2024Q3); (ref 2023Q4); pooled DiD, SE, t; also min coefficient, t range, pre-trend chi2 and df",
         [e1["mean"], e3["mean"], b_t, se_t, t_t, e1["minc"], e1["tmin"], e1["tmax"], e1["chi2"], e1["k"]])

first 2024Q1, ref 2024Q3: post mean -0.56 (min -0.68, t -2.5..-1.5); pre-trend chi2 1.3 on 2 df


first 2023Q1, ref 2024Q3: post mean -0.56 (min -0.68, t -2.6..-1.5); pre-trend chi2 4.2 on 6 df
first 2023Q1, ref 2023Q4: post mean -0.47 (min -0.60, t -2.6..-1.3); pre-trend chi2 2.3 on 3 df


pooled DiD: -0.539 (SE 0.264, t -2.04) -> -42%


### Claim N113: a random set of eight unexposed projects falls that far about 7% of the time (placebo)
For every DPG in the sample: mean core contributors 2023Q1-2024Q3 ("before") and 2025Q1-2026Q3 ("after"). Draw 8 DPGs at random
from the unexposed ones, 10,000 times, and record the change in the mean after / mean before. How often is the fall 40% or more?
**Randomised: the generator is `numpy.random.default_rng(0)` (FIXED seed); each draw uses `DataFrame.sample(8, random_state=<integer from that generator>)`.**
The same seed, numpy and pandas versions reproduce 7.3% exactly; with different versions expect 7.3% plus or minus about 0.3
points (the standard error of a share estimated from 10,000 draws is 0.26 points).

In [33]:
def pre_post(sample_col="in_joint_test", exp_col="exposed"):
    # per DPG: mean core before (2023Q1-2024Q3) and after (2025Q1 on), with its exposure flag
    ids = set(smp.loc[smp[sample_col].astype(str) == "True", "dpgid"])
    flag = flags.set_index("dpgid")[exp_col]
    x = sq[sq.dpgid.isin(ids)].copy(); x["ex"] = x.dpgid.map(flag); x = x[x.ex.notna()]
    pre = x[x.quarter <= "2024Q3"].groupby("dpgid").core_team_dpg.mean()
    post = x[x.quarter >= "2025Q1"].groupby("dpgid").core_team_dpg.mean()
    out = pd.DataFrame({"pre": pre, "post": post}).dropna()
    out["ex"] = out.index.map(flag)
    return out

pp0 = pre_post()
unexp = pp0[pp0.ex == 0]                                        # the pool the random sets are drawn from
rng_pl = np.random.default_rng(0)                               # FIXED seed
chg = []
for _ in range(10000):                                          # 10,000 draws of 8 unexposed DPGs
    k8 = unexp.sample(8, random_state=int(rng_pl.integers(1e9)))
    chg.append(k8.post.mean() / k8.pre.mean() - 1)
chg = np.array(chg)
share_40 = (chg <= -0.40).mean() * 100; p5 = np.percentile(chg, 5) * 100
print(f"{len(unexp)} unexposed DPGs in the pool | share of random sets with a fall of 40% or more: {share_40:.1f}% | 5th percentile {p5:.1f}%")
record_result("N113", "share of random sets of 8 unexposed DPGs that fall 40% or more (%); 5th percentile (%); 5 (the percentile)", [share_40, p5, 5])

124 unexposed DPGs in the pool | share of random sets with a fall of 40% or more: 7.3% | 5th percentile -45.7%


### Claim N114: under the broader exposure definition the before/after means fall 29% against 4%
`funding_flags.exposed_sens` is the broader definition (35 exposed DPGs in the primary sample). Mean core contributors before
and after, exposed against unexposed.

In [34]:
pps = pre_post("in_joint_test", "exposed_sens")
rows_ = {}
for g_, x in pps.groupby("ex"):
    rows_[int(g_)] = (len(x), x.pre.mean(), x.post.mean(), (x.post.mean() / x.pre.mean() - 1) * 100)
print({k: tuple(round(v, 2) for v in t) for k, t in rows_.items()})
record_result("N114", "broader exposure: change in mean core (%) for exposed and unexposed; number of exposed and unexposed DPGs",
         [rows_[1][3], rows_[0][3], rows_[1][0], rows_[0][0]])

{0: (97, np.float64(7.83), np.float64(7.54), np.float64(-3.68)), 1: (35, np.float64(4.45), np.float64(3.15), np.float64(-29.33))}


### Claim N115: output, measured either way, shows no difference
Outcome: log(1 + minor and major releases + issues closed). Event study against the 2024Q3 baseline and the pooled before/after
difference-in-differences.

In [35]:
o1 = es_summary(pan, "output", "2024Q1", "2024Q3")
b_o, se_o, t_o = did(pan, "output")
print(f"output, ref 2024Q3: post mean {o1['mean']:.2f} (t {o1['tmin']:.1f}..{o1['tmax']:.1f}); pre-trend chi2 {o1['chi2']:.1f} on {o1['k']} df")
print(f"pooled DiD: {b_o:+.3f} (SE {se_o:.3f}, t {t_o:.2f})")
record_result("N115", "output: post mean (ref 2024Q3), t range; pooled DiD, SE, t", [o1["mean"], o1["tmin"], o1["tmax"], b_o, se_o, t_o])

output, ref 2024Q3: post mean -0.06 (t -1.0..0.3); pre-trend chi2 0.6 on 2 df
pooled DiD: +0.048 (SE 0.090, t 0.54)


## 6. Summary of the numbers reproduced here

In [36]:
print_results()

N015  exposed DPGs; with terminated award; with no match
      -> 12, 9, 3
N071  exposed DPGs with panel data
      -> 8
N072  FarmStack core: 2023Q1-Q3 min and max, 2024Q1, 2024Q4, and 2025Q3 on
      -> 12, 14, 7, 2, 0, 3, 1
N073  JemPI core 2024Q1, 2024Q3, and max from 2024Q4 on
      -> 7, 5, 0
N074  Harmony max core in 2023; in 2024; from 2025
      -> 4, 2, 0
N075  AMP core 2024Q2, 2024Q3, 2024Q4, 2025 min and max, 2026Q3
      -> 5, 2, 1, 3, 4, 2
N076  OCP max core from 2024Q1
      -> 0
N077  CommCare core 2024Q1 and 2026Q3
      -> 27, 14
N078  HOT core: 2024 min/max, 2025 min/max, 2026 level
      -> 9, 13, 7, 10, 6, 5, 6
N079  CHT core: min, max, mean, quarters at >=14, quarters 2024Q1..2026Q3
      -> 7, 18, 13.6364, 9, 13, 11
N081  quarter of the drop: 2024Q4
      -> 2024, 4
N080  mean core: exposed before/after, unexposed before/after, % changes, n exposed, n unexposed
      -> 9.14286, 5.28571, 6.59981, 6.31303, -42.1875, -4.34531, 8, 136
N082  reference quarter 2024Q4;